In [ ]:
from abc import ABC, abstractmethod
import uuid

class BankAccountError(Exception):
    """Класс базового исключения для банковских ошибок."""
    pass

class AccountFrozenError(BankAccountError):
    """Операция над замороженным счётом."""
    pass


class AccountClosedError(BankAccountError):
    """Операция над закрытым счётом."""
    pass


class InvalidOperationError(BankAccountError):
    """Некорректная операция"""
    pass


class InsufficientFundsError(BankAccountError):
    """Недостаточно средств на счёте для операции."""
    pass


class AbstractAccount(ABC):
    """Абстрактный банковский счёт"""

    @abstractmethod
    def deposit(self, amount):
        """Пополнить счёт."""
        pass

    
    @abstractmethod
    def withdraw(self, amount):
        """Снять деньги со счёта."""
        pass

    @abstractmethod
    def get_account_info(self):
        """Вернуть информацию о счёте."""
        pass

ALLOWED_CURRENCIES = ("RUB", "USD", "EUR", "KZT", "CNY")
ALLOWED_STATUSES = ("active", "frozen", "closed")


class BankAccount(AbstractAccount):
    def __init__(self,owner,balance = 0,currency = 'RUB',account_id = None,status = 'active'):

        if not isinstance(owner, str) or not owner.strip():
            raise InvalidOperationError('Владелец должен быть непустой строкой')
        self.owner = owner.strip()

        if not isinstance(balance, (int, float)):
            raise InvalidOperationError(
                f'Баланс должен быть числом, получено {type(balance).__name__}'
            )
        if balance < 0:
            raise InvalidOperationError(
                f'Баланс не может быть отрицательным, получено {balance!r}'
            )
        self._balance = balance

        
        if status not in ALLOWED_STATUSES:
            raise InvalidOperationError(f'Недопустимый статус: {status}')
        self._status = status

        if currency not in ALLOWED_CURRENCIES:
            raise InvalidOperationError(
                f'Недопустимая валюта: {currency}. '
                f'Разрешены: {", ".join(ALLOWED_CURRENCIES)}'
            )
        self._currency = currency

        if account_id is None:
            self.account_id = str(uuid.uuid4())[:8]
        else:
            self.account_id = account_id
        
    def __str__(self):
        short_id = str(self.account_id)[-4:] 
        return (
            f"BankAccount(owner={self.owner}, "
            f"id=...{short_id}, "
            f"status={self._status}, "
            f"balance={self._balance} {self._currency})"
        )

    def _check_active(self):
        if self._status == 'frozen':
            raise AccountFrozenError(f'Счет {self.account_id} заморожен')
        elif self._status == 'closed':
            raise AccountClosedError(f'Счет {self.account_id} закрыт')                    

    def _validate_amount(self,amount):
        if isinstance(amount, bool) or not isinstance(amount,(float,int)):
            raise InvalidOperationError(f'Сумма должна быть числом, получено {amount}')
        if amount <= 0:
           raise InvalidOperationError(f'Сумма должна быть положительной, получено {amount}')
        
    def deposit(self, amount):
        self._check_active()
        self._validate_amount(amount)
        self._balance += amount

    def withdraw(self, amount):
        self._check_active()
        self._validate_amount(amount)
        if amount > self._balance:
            raise InsufficientFundsError('Недостаточно средств на счёте для операции')
        self._balance -= amount

    def get_account_info(self):
        """Вернуть информацию о счёте."""
        return f'Счет {self.account_id} со статусом {self._status} имеет {self._balance} {self._currency}'

In [17]:
# 1. Создание
acc = BankAccount("Иван", 1000, "RUB")
print("Создан:", acc)

# 2. Пополнение
acc.deposit(500)
assert acc._balance == 1500
print("Пополнение:", acc._balance)

# 3. Снятие
acc.withdraw(200)
assert acc._balance == 1300
print("Снятие:", acc._balance)

# 4. Недостаточно средств
try:
    acc.withdraw(99999)
    print("Не упало")
except InsufficientFundsError as e:
    print(f"InsufficientFundsError: {e}")

# 5. Плохая сумма
for bad in [0, -100, "abc"]:
    try:
        acc.deposit(bad)
        print(f"Не упало на {bad}")
    except InvalidOperationError:
        pass
print("Валидация суммы работает")

# 6. Замороженный счёт
frozen = BankAccount("Пётр", 500, status="frozen")
try:
    frozen.deposit(100)
    print("Замороженный не заблокирован")
except AccountFrozenError as e:
    print(f"AccountFrozenError: {e}")

# 7. Закрытый счёт
closed = BankAccount("Анна", 0, status="closed")
try:
    closed.deposit(100)
    print("Закрытый не заблокирован")
except AccountClosedError as e:
    print(f"AccountClosedError: {e}")

# 8. Плохая валюта
try:
    BankAccount("Иван", currency="rub")
    print("'rub' прошёл")
except InvalidOperationError as e:
    print(f"Строгая валюта: {e}")


Создан: BankAccount(owner=Иван, id=...ef81, status=active, balance=1000 RUB)
Пополнение: 1500
Снятие: 1300
InsufficientFundsError: Недостаточно средств на счёте для операции
Валидация суммы работает
AccountFrozenError: Счет 42224aae заморожен
AccountClosedError: Счет 67764ad0 закрыт
Строгая валюта: Недопустимая валюта: rub. Разрешены: RUB, USD, EUR, KZT, CNY


In [19]:
from day1 import (
    BankAccount,
    InvalidOperationError,
    InsufficientFundsError,
    AccountFrozenError,
    AccountClosedError,
)
class SavingsAccount(BankAccount):
    def __init__(self, owner, balance=0, currency='RUB', account_id=None, status='active',min_balance = 100, monthly_rate=0.005):

        if not isinstance(min_balance,(int,float)) or isinstance(min_balance,bool) or min_balance < 0:
            raise InvalidOperationError(
                f'Минимальный остаток должен быть неотрицательным числом, получено {min_balance}'
            )
        
        if not isinstance(monthly_rate, (int, float)) or isinstance(monthly_rate,bool) or monthly_rate < 0:
            raise InvalidOperationError(
                f'Ставка должна быть неотрицательным числом, получено {monthly_rate}'
            )

        if min_balance > balance:
            raise InvalidOperationError(
                f'Начальный баланс ({balance}) не может быть меньше минимального остатка ({min_balance})'
            )

        super().__init__(owner, balance, currency, account_id, status)
        self._min_balance = min_balance
        self._monthly_rate = monthly_rate

    def __str__(self):
        return (
            f"{super().__str__()}, "
            f"min_balance={self._min_balance}, "
            f"monthly_rate={self._monthly_rate * 100}%)"
        )

    def apply_monthly_interest(self):
        self._check_active()
        interest = self._balance * self._monthly_rate
        self._balance += interest
        return interest

    def withdraw(self, amount):
        if self._balance - amount < self._min_balance:
            raise InvalidOperationError(
                f'Нельзя снять {amount}: баланс станет {self._balance - amount}, '
                f'а минимум — {self._min_balance}'
            )
        super().withdraw(amount)

    def get_account_info(self):
        """Вернуть информацию о счёте."""

        return (f'{super().get_account_info()}, Накопительный счет:'
            f' мин.остаток = {self._min_balance}, '
            f'ставка = {self._monthly_rate * 100}% в месяц')
    
    
class PremiumAccount(BankAccount):
    def __init__(self, owner, balance=0, currency='RUB', account_id=None, status='active',overdraft_limit  = 0,operation_fee = 0):
        
        if (not isinstance(overdraft_limit, (int, float))
                or isinstance(overdraft_limit, bool)
                or overdraft_limit < 0):
            raise InvalidOperationError(
                f'Лимит овердрафта должен быть неотрицательным числом, получено {overdraft_limit}'
            )
        
        if (not isinstance(operation_fee, (int, float))
                or isinstance(operation_fee, bool)
                or operation_fee < 0):
            raise InvalidOperationError(
                f'Комиссия должна быть неотрицательным числом, получено {operation_fee}'
            )
        super().__init__(owner, balance, currency, account_id, status) 

        self._overdraft_limit = overdraft_limit
        self._operation_fee = operation_fee

    def __str__(self):
            return (
                f"{super().__str__()}, "
                f"overdraft_limit={self._overdraft_limit}, "
                f"operation_fee={self._operation_fee})"
            )
    
    def withdraw(self, amount):
        new_balance = self._balance - amount - self._operation_fee
        if new_balance < -self._overdraft_limit:
            raise InsufficientFundsError(
            f'Превышен лимит овердрафта: баланс станет {new_balance}, а минимум — {-self._overdraft_limit}'
        )
        self._check_active()
        self._validate_amount(amount)
        self._balance -= amount + self._operation_fee

    def get_account_info(self):
        return (
        f'{super().get_account_info()}, '
        f'Премиум: овердрафт={self._overdraft_limit}, '
        f'комиссия={self._operation_fee} {self._currency}'
    )

    
class InvestmentAccount(BankAccount):
    ALLOWED_ASSETS = ("stocks", "bonds", "etf")

    def __init__(self, owner, balance=0, currency='RUB', account_id=None, status='active',portfolio = None):
        super().__init__(owner, balance, currency, account_id, status)

        if portfolio is None:
            portfolio = {}

        if not isinstance(portfolio,dict):
            raise InvalidOperationError(
                f'Портфель должен быть словарём, получено {type(portfolio).__name__}'
            )

        for active_type,amount in portfolio.items():
            if active_type not in self.ALLOWED_ASSETS:
                raise InvalidOperationError(
                    f'Недопустимый тип актива: {active_type}. '
                    f'Разрешены: {", ".join(self.ALLOWED_ASSETS)}'
                )
            if (not isinstance(amount, (int, float))
                    or isinstance(amount, bool)
                    or amount < 0):
                raise InvalidOperationError(
                    f'Сумма актива {active_type} должна быть неотрицательным числом, получено {amount}'
                )

        self._portfolio = dict(portfolio)

    def __str__(self):
        portfolio_str = ", ".join(
        f"{asset}={amount}" for asset, amount in self._portfolio.items()
        ) or "пусто"
        return (
        f"{super().__str__()}, "
        f"portfolio=[{portfolio_str}])"
        )

    def project_yearly_growth(self, growth_rates):
        """Прогноз годового прироста портфеля."""
        if not isinstance(growth_rates, dict):
            raise InvalidOperationError(
                f'growth_rates должен быть словарём, получено {type(growth_rates).__name__}'
            )

        growth = {}
        for active_type,amount in self._portfolio.items():
            if active_type not in growth_rates:
                raise InvalidOperationError(
                    f'Нет ставки для актива {active_type} в growth_rates'
                )
            
            rate = growth_rates[active_type]
            if (not isinstance(rate, (int, float))
                or isinstance(rate, bool)
                or rate < 0):
                raise InvalidOperationError(
                    f'Ставка для {active_type} должна быть неотрицательным числом, получено {rate}'
                )

            growth[active_type] = amount * rate

        growth['total'] = sum(growth.values())
        return  growth

    def get_account_info(self):
        portfolio_str = ", ".join(
            f"{active_type}={amount}" for active_type, amount in self._portfolio.items()
        ) or "пусто"
        return (
            f'{super().get_account_info()}, '
            f'Портфель: [{portfolio_str}]'
        )

if __name__ == "__main__":
    print()
    print("1. SAVINGS ACCOUNT")
    print()
    s = SavingsAccount("Иван", balance=1000, min_balance=200, monthly_rate=0.05)
    print(s)
    print(s.get_account_info())

    interest = s.apply_monthly_interest()
    print(f"Проценты: начислено {interest}, баланс = {s._balance}")

    s.withdraw(500)
    print(f"Снятие 500: баланс = {s._balance}")

    try:
        s.withdraw(400)  
        print("Должно было упасть")
    except InvalidOperationError as e:
        print(f"Ошибка: {e}")
        print(f"Баланс не изменился: {s._balance}")

    print()
    print("2. PREMIUM ACCOUNT")
    print()
    p = PremiumAccount("Олег", balance=1000, overdraft_limit=500, operation_fee=10)
    print(p)
    print(p.get_account_info())

    p.withdraw(300)
    print(f"Снятие 300 +комиссия 10: баланс = {p._balance}")

    p.withdraw(1000)
    print(f"Снятие 1000 в овердрафт:   баланс = {p._balance}")

    try:
        p.withdraw(300)   
        print("Должно было упасть")
    except InsufficientFundsError as e:
        print(f"Ошибка: {e}")

    p.deposit(500)
    print(f"Пополнение 500 без комиссии: баланс = {p._balance}")

    
    print()
    print("3. INVESTMENT ACCOUNT")
    print()

    inv = InvestmentAccount(
        "Анна",
        balance=10000,
        portfolio={"stocks": 3000, "bonds": 2000, "etf": 1000},
    )
    print(inv)
    print(inv.get_account_info())

    rates = {"stocks": 0.10, "bonds": 0.04, "etf": 0.07}
    growth = inv.project_yearly_growth(rates)
    print(f"Прогноз роста: {growth}")

    inv.withdraw(500)
    print(f"Снятие 500: баланс = {inv._balance}")
    print(f"Портфель после снятия: {inv._portfolio}")

    


1. SAVINGS ACCOUNT

SavingsAccount(owner=Иван, id=...a618, status=active, balance=1000 RUB, min_balance=200, monthly_rate=5.0%)
Счет fd7ba618 со статусом active имеет 1000 RUB, Накопительный счет: мин.остаток = 200, ставка = 5.0% в месяц
Проценты: начислено 50.0, баланс = 1050.0
Снятие 500: баланс = 550.0
Ошибка: Нельзя снять 400: баланс станет 150.0, а минимум — 200
Баланс не изменился: 550.0

2. PREMIUM ACCOUNT

PremiumAccount(owner=Олег, id=...f65e, status=active, balance=1000 RUB, overdraft_limit=500, operation_fee=10)
Счет dd7ef65e со статусом active имеет 1000 RUB, Премиум: овердрафт=500, комиссия=10 RUB
Снятие 300 +комиссия 10: баланс = 690
Снятие 1000 в овердрафт:   баланс = -320
Ошибка: Превышен лимит овердрафта: баланс станет -630, а минимум — -500
Пополнение 500 без комиссии: баланс = 180

3. INVESTMENT ACCOUNT

InvestmentAccount(owner=Анна, id=...960f, status=active, balance=10000 RUB, portfolio=[stocks=3000, bonds=2000, etf=1000])
Счет 9371960f со статусом active имеет 10

In [ ]:
import re
import uuid
from datetime import datetime, time   # для Bank пригодится

from day1 import (
    BankAccount,
    InvalidOperationError,
    InsufficientFundsError,
    AccountFrozenError,
    AccountClosedError,
)
from day2 import SavingsAccount, PremiumAccount, InvestmentAccount

EMAIL_REGEX = re.compile(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")
PHONE_PATTERN = re.compile(r"^[\+\d\s\-\(\)\.]+$")

class Client:

    STATUS_ACTIVE = "active"
    STATUS_BLOCKED = "blocked"
    STATUS_SUSPICIOUS = "suspicious"

    def __init__(self, full_name, age, email, phone, password,client_id=None):

        if not isinstance(full_name, str) or not full_name.strip():
            raise InvalidOperationError(
                'ФИО должно быть непустой строкой'
            )
        self.full_name = full_name.strip()

        if not isinstance(age, int) or isinstance(age, bool):
            raise InvalidOperationError(
                f'Возраст клиента должен быть целочисленным, получено {age}'
            )
        if age < 18:
            raise InvalidOperationError(
                f'Клиент должен быть старше 18 лет, получено {age}'
            )
        self.age = age

        if not isinstance(email, str):
            raise InvalidOperationError(
                f'Email должен быть строкой, получено {type(email).__name__}'
            )
        email = email.strip()
        if not EMAIL_REGEX.match(email):
            raise InvalidOperationError(f'Некорректный email: {email}')
        self.email = email

        if not isinstance(phone, str):
            raise InvalidOperationError(
                f'Телефон должен быть строкой, получено {type(phone).__name__}'
            )
        
        if not PHONE_PATTERN.match(phone):
            raise InvalidOperationError(
                f'Телефон содержит недопустимые символы: {phone}. '
                f'Разрешены: цифры, +, -, пробелы, скобки, точки'
            )
        digits = re.sub(r"\D", "", phone)   
        if not (10 <= len(digits) <= 15):
            raise InvalidOperationError(
                f'Телефон должен содержать 10–15 цифр, получено {len(digits)}: {phone!r}'
            )
        self.phone = phone.strip()

        if not isinstance(password, str) or len(password) < 4:
            raise InvalidOperationError('Пароль должен быть строкой минимум из 4 символов')
            
        
        self.password = password

        if client_id is None:
            self.client_id = str(uuid.uuid4())[:8]
        else:
            if not isinstance(client_id, str) or not client_id.strip():
                raise InvalidOperationError(
                    'ID должен быть непустой строкой'
                )
            self.client_id = client_id.strip()

        self.status = self.STATUS_ACTIVE
        self.account_ids = []
        self.failed_attempts = 0
        self.suspicious_actions = []

    def add_account(self, account_id):
        """Добавить ID счёта к клиенту."""
        if not isinstance(account_id, str) or not account_id.strip():
            raise InvalidOperationError('account_id должен быть непустой строкой')
        if account_id in self.account_ids:
            raise InvalidOperationError(f'Счёт {account_id!r} уже привязан к клиенту')
        self.account_ids.append(account_id)

    def remove_account(self, account_id):
        """Убрать ID счёта у клиента."""
        if account_id not in self.account_ids:
            raise InvalidOperationError(f'Счёт {account_id!r} не привязан к клиенту')
        self.account_ids.remove(account_id)


class Bank:

    ACCOUNT_TYPES = {
    "basic": BankAccount,
    "savings": SavingsAccount,
    "premium": PremiumAccount,
    "investment": InvestmentAccount,
    }

    _allow_operations_24h = False

    def __init__(self,name):
        if not isinstance(name, str) or not name.strip():
            raise InvalidOperationError('Название банка должно быть непустой строкой')
        self.name = name.strip()

        self._clients = {}      
        self._accounts = {}     
        self._max_failed_attempts = 3

    def _get_client(self, client_id):
        if client_id not in self._clients:
            raise InvalidOperationError(f'Клиент с ID {client_id} не найден')
        return self._clients[client_id]

    def _get_account(self, account_id):
        if account_id not in self._accounts:
            raise InvalidOperationError(f'Счёт с ID {account_id} не найден')
        return self._accounts[account_id]

    def add_client(self, client):
        """Добавить клиента в банк."""
        if not isinstance(client, Client):
            raise InvalidOperationError(f'Ожидается объект Client, получено {type(client).__name__}')
        
        if client.client_id in self._clients:
            raise InvalidOperationError(f'Клиент с ID {client.client_id} уже зарегистрирован')
        
        self._clients[client.client_id] = client
        
        return client.client_id


        

    def _check_working_hours(self):
        if self._allow_operations_24h:
            return   # тестовый режим 
        now = datetime.now().time()
        if time(0, 0) <= now < time(5, 0):
            raise InvalidOperationError('Операции запрещены с 00:00 до 05:00')

    def open_account(self, client_id, account_type="basic", **kwargs):
        """Открытие счёта для клиента"""
        self._check_working_hours()

        client = self._get_client(client_id)

        if client.status == Client.STATUS_BLOCKED:
            raise InvalidOperationError(f'Клиент {client.full_name} заблокирован — операция невозможна')

        if account_type not in self.ACCOUNT_TYPES:
            raise InvalidOperationError(f'Неизвестный тип счёта: {account_type}. Доступны: {", ".join(self.ACCOUNT_TYPES.keys())}')

        AccountClass = self.ACCOUNT_TYPES[account_type]
        account = AccountClass(owner=client.full_name, **kwargs)

        client.add_account(account.account_id)
        self._accounts[account.account_id] = account

        return account

    def close_account(self, account_id):
        """Закрыть счёт."""
        self._check_working_hours()

        account = self._get_account(account_id)

        if account._status == "closed":
            raise InvalidOperationError(f'Счёт {account_id} уже закрыт')

        account._status = "closed"
        return account

    def freeze_account(self, account_id):
        """Заморозить счёт."""
        self._check_working_hours()

        account = self._get_account(account_id)

        if account._status == "closed":
            raise InvalidOperationError(f'Нельзя заморозить закрытый счёт {account_id}')

        if account._status == "frozen":
            raise InvalidOperationError(f'Счёт {account_id} уже заморожен')

        account._status = "frozen"
        return account

    def unfreeze_account(self, account_id):
        """Разморозить счёт."""
        self._check_working_hours()

        account = self._get_account(account_id)

        if account._status != "frozen":
            raise InvalidOperationError(f'Счёт {account_id} не заморожен (текущий статус: {account._status})')

        account._status = "active"
        return account

    def authenticate_client(self, client_id, password):
        """Проверить пароль. Блокирует клиента после 3 неудач."""
        client = self._get_client(client_id)

        if client.status == Client.STATUS_BLOCKED:
            raise InvalidOperationError(f'Клиент {client.full_name} заблокирован. ')

        if password != client.password:
            client.failed_attempts += 1
            client.suspicious_actions.append({
                "time": datetime.now().isoformat(),
                "reason": "Неверный пароль",
                "attempt": client.failed_attempts,
            })
            
            if len(client.suspicious_actions) >= 2:
                client.status = Client.STATUS_SUSPICIOUS

            remaining = self._max_failed_attempts - client.failed_attempts

            if client.failed_attempts >= self._max_failed_attempts:
                client.status = Client.STATUS_BLOCKED
                raise InvalidOperationError(f'Клиент {client.full_name} заблокирован после {self._max_failed_attempts} неудачных попыток')

            raise InvalidOperationError(f'Неверный пароль. Осталось попыток: {remaining}')

        client.failed_attempts = 0
        return True

    def search_accounts(self, owner=None, status=None):
        """Поиск счетов по владельцу или статусу."""
        results = []
        for account in self._accounts.values():
            if owner is not None and owner.lower() not in account.owner.lower():
                continue
            if status is not None and account._status != status:
                continue
            results.append(account)
        return results

    def get_total_balance(self, client_id=None):
        """Общий баланс."""
        if client_id is not None:
            client = self._get_client(client_id)
            accounts = [self._accounts[aid] for aid in client.account_ids if aid in self._accounts]

        else:
            accounts = list(self._accounts.values())

        return sum(acc._balance for acc in accounts)

    def get_clients_ranking(self):
        """Рейтинг клиентов по общей сумме балансов."""
        ranking = []
        for client in self._clients.values():
            total = sum(self._accounts[aid]._balance for aid in client.account_ids if aid in self._accounts)
            ranking.append((client, total))

        ranking.sort(key=lambda pair: pair[1], reverse=True)
        return ranking

In [ ]:
import uuid
from datetime import datetime

from day1 import (
    InvalidOperationError,
    InsufficientFundsError,
)
from day2 import PremiumAccount
from day3 import Bank, Client

class Transaction:

    STATUS_PENDING = "pending"
    STATUS_COMPLETED = "completed"
    STATUS_FAILED = "failed"
    STATUS_CANCELLED = "cancelled"

    TYPE_TRANSFER = "transfer"
    TYPE_DEPOSIT = "deposit"
    TYPE_WITHDRAWAL = "withdrawal"
    ALLOWED_TYPES = (TYPE_TRANSFER, TYPE_DEPOSIT, TYPE_WITHDRAWAL)

    PRIORITY_HIGH = "high"
    PRIORITY_NORMAL = "normal"

    ALLOWED_CURRENCIES = ("RUB", "USD", "EUR", "KZT", "CNY")

    def __init__(self, transaction_type, amount, currency,
                 sender_id=None, receiver_id=None,
                 transaction_id=None, priority=None):
        
        if transaction_type not in self.ALLOWED_TYPES:
            raise InvalidOperationError( 
                f'Недопустимый тип транзакции: {transaction_type}. '
                f'Разрешены: {", ".join(self.ALLOWED_TYPES)}'
            )
        self.transaction_type = transaction_type

        if (not isinstance(amount, (int, float))
            or isinstance(amount, bool)
            or amount <= 0):
            raise InvalidOperationError(f'Сумма должна быть положительным числом, получено {amount}')
        self.amount = amount

        if not isinstance(currency, str):
            raise InvalidOperationError(f'Валюта должна быть строкой, получено {type(currency).__name__}')
        currency = currency.strip().upper()
        if currency not in self.ALLOWED_CURRENCIES:
            raise InvalidOperationError(f'Недопустимая валюта: {currency}. Разрешены: {", ".join(self.ALLOWED_CURRENCIES)}')
        self.currency = currency

        if sender_id is not None and (not isinstance(sender_id, str) or not sender_id.strip()):
            raise InvalidOperationError(f'sender_id должен быть непустой строкой, получено {sender_id}')
        if receiver_id is not None and (not isinstance(receiver_id, str) or not receiver_id.strip()):
            raise InvalidOperationError(f'receiver_id должен быть непустой строкой, получено {receiver_id}')

        if transaction_type == self.TYPE_TRANSFER:
            if not sender_id or not receiver_id:
                raise InvalidOperationError('Для transfer нужны и sender_id, и receiver_id')
        elif transaction_type == self.TYPE_DEPOSIT:
            if not receiver_id:
                raise InvalidOperationError('Для deposit нужен receiver_id')
            if sender_id:
                raise InvalidOperationError('Для deposit sender_id должен быть None')
        elif transaction_type == self.TYPE_WITHDRAWAL:
            if not sender_id:
                raise InvalidOperationError('Для withdrawal нужен sender_id')
            if receiver_id:
                raise InvalidOperationError('Для withdrawal receiver_id должен быть None')
        self.sender_id = sender_id.strip() if sender_id else None
        self.receiver_id = receiver_id.strip() if receiver_id else None
        

        if transaction_id is None:
            self.transaction_id = str(uuid.uuid4())[:8]
        else:
            if not isinstance(transaction_id, str) or not transaction_id.strip():
                raise InvalidOperationError('transaction_id должен быть непустой строкой')
            self.transaction_id = transaction_id.strip()

        if priority is None:
            priority = self.PRIORITY_NORMAL
        if priority not in (self.PRIORITY_HIGH, self.PRIORITY_NORMAL):
            raise InvalidOperationError(f'Неизвестный приоритет: {priority}. Разрешены: {self.PRIORITY_HIGH}, {self.PRIORITY_NORMAL}')
        self.priority = priority

        self.fee = 0.0                    
        self.status = self.STATUS_PENDING
        self.failure_reason = None
        self.created_at = datetime.now()
        self.processed_at = None

    def __str__(self):
        short_id = f'...{str(self.transaction_id)[-4:]}'
        sender = f"...{self.sender_id[-4:]}" if self.sender_id else "—"
        receiver = f"...{self.receiver_id[-4:]}" if self.receiver_id else "—"
        return (
            f"Transaction(id={short_id}, "
            f"type={self.transaction_type}, "
            f"amount={self.amount} {self.currency}, "
            f"fee={self.fee}, "
            f"status={self.status}, "
            f"from={sender}, to={receiver})"
        )    

    def mark_completed(self):
        """Пометить транзакцию выполненной."""
        if self.status != self.STATUS_PENDING:
            raise InvalidOperationError(f'Нельзя завершить транзакцию в статусе {self.status}. Ожидается: pending')
        self.status = self.STATUS_COMPLETED
        self.processed_at = datetime.now()

    def mark_failed(self, reason):
        """Пометить транзакцию упавшей с указанием причины."""
        if not isinstance(reason, str) or not reason.strip():
            raise InvalidOperationError('Причина отказа должна быть непустой строкой')
        if self.status != self.STATUS_PENDING:
            raise InvalidOperationError(f'Нельзя пометить failed транзакцию в статусе {self.status}')
        
        self.status = self.STATUS_FAILED
        self.failure_reason = reason.strip()
        self.processed_at = datetime.now()

    def mark_cancelled(self):
        """Отменить транзакцию."""
        if self.status != self.STATUS_PENDING:
            raise InvalidOperationError(f'Нельзя отменить транзакцию в статусе {self.status}. Отменить можно только pending.')
        self.status = self.STATUS_CANCELLED
        self.processed_at = datetime.now()

class TransactionQueue:
    def __init__(self):
        self._high = []              
        self._normal = []            
        self._deferred = []          
        self._cancelled = []   

    def __str__(self):
        return (
            f"TransactionQueue(high={len(self._high)}, "
            f"normal={len(self._normal)}, "
            f"deferred={len(self._deferred)}, "
            f"cancelled={len(self._cancelled)})"
        )   

    def is_empty(self):
        return not self._high and not self._normal


    def __len__(self):
        return len(self._high) + len(self._normal) + len(self._deferred)   

    def _promote_deferred(self):
        """Переместить готовые отложенные в основные очереди."""
        now = datetime.now()
        still_deferred = []

        for execute_after, transaction in self._deferred:
            if now >= execute_after:
                if transaction.priority == Transaction.PRIORITY_HIGH:
                    self._high.append(transaction)
                else:
                    self._normal.append(transaction)
            else:
                still_deferred.append((execute_after, transaction))

        self._deferred = still_deferred

    def add(self, transaction):
        """Добавить транзакцию в очередь по её приоритету."""
        if not isinstance(transaction, Transaction):
            raise InvalidOperationError(f'Ожидается Transaction, получено {type(transaction).__name__}')
        if transaction.status != Transaction.STATUS_PENDING:
            raise InvalidOperationError(f'Нельзя добавить транзакцию в статусе {transaction.status}.Ожидается: pending')
        if transaction.priority == Transaction.PRIORITY_HIGH:
            self._high.append(transaction)
        else:
            self._normal.append(transaction)

    def get_next(self):
        """Взять следующую транзакцию"""
        self._promote_deferred()

        if self._high:
            return self._high.pop(0)
        if self._normal:
            return self._normal.pop(0)
        return None

    def cancel(self, transaction_id):
        """Отменить транзакцию"""
        for queue in (self._high, self._normal):
            for i, t in enumerate(queue):
                if t.transaction_id == transaction_id:
                    t.mark_cancelled()
                    queue.pop(i)
                    self._cancelled.append(t)
                    return t

        for i, (_, t) in enumerate(self._deferred):
            if t.transaction_id == transaction_id:
                t.mark_cancelled()
                self._deferred.pop(i)
                self._cancelled.append(t)
                return t

        raise InvalidOperationError( f'Транзакция {transaction_id} не найдена в очереди')

    def defer(self, transaction_id, execute_after):
        """Отложить транзакцию до указанного времени."""
        if not isinstance(execute_after, datetime):
            raise InvalidOperationError(f'execute_after должен быть datetime, получено {type(execute_after).__name__}')
        if execute_after <= datetime.now():
            raise InvalidOperationError(f'execute_after должен быть в будущем, получено {execute_after}')

        for queue in (self._high, self._normal):
            for i, t in enumerate(queue):
                if t.transaction_id == transaction_id:
                    queue.pop(i)
                    self._deferred.append((execute_after, t))
                    return t

        raise InvalidOperationError(f'Транзакция {transaction_id} не найдена для откладывания')

class TransactionProcessor:

    DEFAULT_FEE_RATE = 0.01               
    PREMIUM_FEE_RATE = 0.005              
    MAX_RETRIES = 3                       

    def __init__(self, bank):
        
        if not isinstance(bank, Bank):
            raise InvalidOperationError(f'Ожидается Bank, получено {type(bank).__name__}')
        self.bank = bank                  
        self.error_log = []            
        self.rates = {
            "RUB": 1.0,      # через рубль
            "USD": 90.0,     
            "EUR": 100.0,    
            "KZT": 0.2,      
            "CNY": 12.5,     
        }

    def process_all(self, queue):
        """Обработать все транзакции из очереди."""
        processed = []
        while not queue.is_empty():
            transaction = queue.get_next()
            if transaction is None:
                break
            self.execute(transaction)
            processed.append(transaction)
        return processed

    def execute(self, transaction):
        """Выполнить одну транзакцию."""
        if not isinstance(transaction, Transaction):
            raise InvalidOperationError(f'Ожидается Transaction, получено {type(transaction).__name__}')
        if transaction.status != Transaction.STATUS_PENDING:
            raise InvalidOperationError(f'Нельзя выполнить транзакцию в статусе {transaction.status}')

        # Расчёт комиссии
        transaction.fee = self._calculate_fee(transaction)

        # Повторная попытка
        try:
            self._execute_with_retry(transaction)
            transaction.mark_completed()
        except Exception as e:
            transaction.mark_failed(str(e))
            self.error_log.append({ # фиксация ошибок
                "transaction_id": transaction.transaction_id,
                "error": str(e),
                "type": type(e).__name__,
            })

    def _calculate_fee(self, transaction):
        """Рассчет комиссий"""
        if transaction.transaction_type != Transaction.TYPE_TRANSFER:
            return 0.0

        # Проверяем, есть ли у отправителя премиум акк
        sender = self.bank._get_account(transaction.sender_id)
        rate = self.DEFAULT_FEE_RATE
        if isinstance(sender, PremiumAccount):
            rate = self.PREMIUM_FEE_RATE

        return transaction.amount * rate

    def _convert(self, amount, from_currency, to_currency):
        """Конвертировать сумму из одной валюты в другую."""
        if from_currency == to_currency:
            return amount

        if from_currency not in self.rates or to_currency not in self.rates:
            raise InvalidOperationError(f'Нет курса для перевода из {from_currency} в {to_currency}')

        amount_in_rub = amount * self.rates[from_currency]
        return amount_in_rub / self.rates[to_currency]

    def _execute_with_retry(self, transaction):
        """Повторные попытки"""
        for attempt in range(1, self.MAX_RETRIES + 1):
            try:
                self._dispatch(transaction)
                return
            except (InsufficientFundsError, InvalidOperationError):
                raise    # бизнес-ошибки — сразу
            except Exception:
                if attempt == self.MAX_RETRIES:
                    raise
                continue

    def _dispatch(self, transaction):
        """Направить транзакцию в нужный обработчик."""
        if transaction.transaction_type == Transaction.TYPE_TRANSFER:
            self._execute_transfer(transaction)
        elif transaction.transaction_type == Transaction.TYPE_DEPOSIT:
            self._execute_deposit(transaction)
        elif transaction.transaction_type == Transaction.TYPE_WITHDRAWAL:
            self._execute_withdrawal(transaction)

    def _execute_deposit(self, transaction):
        """Пополнение счёта."""
        account = self.bank._get_account(transaction.receiver_id)

        if account._currency != transaction.currency:
            amount = self._convert(
                transaction.amount,
                transaction.currency,
                account._currency,
            )
        else:
            amount = transaction.amount

        account.deposit(amount)

    def _execute_withdrawal(self, transaction):
        """Снятие со счёта."""
        account = self.bank._get_account(transaction.sender_id)

        if account._currency != transaction.currency:
            amount = self._convert(
                transaction.amount,
                transaction.currency,
                account._currency,
            )
        else:
            amount = transaction.amount

        if not isinstance(account, PremiumAccount):
            if account._balance - amount < 0:
                raise InsufficientFundsError(
                    f'Нельзя уйти в минус (счёт {account.account_id}). '
                    f'Баланс: {account._balance}, снятие: {amount}'
                )

        account.withdraw(amount)

    def _execute_transfer(self, transaction):
        """Перевод между счетами."""
        sender = self.bank._get_account(transaction.sender_id)
        receiver = self.bank._get_account(transaction.receiver_id)

        total_from_sender = transaction.amount + transaction.fee

        if sender._currency != receiver._currency:
            received = self._convert(
                transaction.amount,
                transaction.currency,
                receiver._currency,
            )
        else:
            received = transaction.amount

        if not isinstance(sender, PremiumAccount):
            if sender._balance - total_from_sender < 0:
                raise InsufficientFundsError(
                    f'Недостаточно средств: баланс {sender._balance}, '
                    f'нужно {total_from_sender}, включая комиссию {transaction.fee}'
                )

        if receiver._status != "active":
            raise InvalidOperationError(f'Счёт получателя {receiver.account_id} не активен: {receiver._status}')

        sender.withdraw(total_from_sender)
        receiver.deposit(received)

    
    

In [ ]:
import json
from datetime import datetime,timedelta,time
from day4 import Transaction
from day1 import InvalidOperationError


class AuditLog:
    """Журнал аудита"""

    LEVEL_INFO = "info"
    LEVEL_WARNING = "warning"
    LEVEL_ERROR = "error"

    ALLOWED_LEVELS = (LEVEL_INFO, LEVEL_WARNING, LEVEL_ERROR)

    def __init__(self, file_path=None):
        """Создать журнал аудита"""
        if file_path is not None:
            if not isinstance(file_path, str) or not file_path.strip():
                raise InvalidOperationError(f'file_path должен быть непустой строкой, получено {file_path}')
            file_path = file_path.strip()

        self._file_path = file_path
        self._events = []

    def __len__(self):
        """Количество событий в памяти"""
        return len(self._events)


    def __str__(self):
        counts = self.count_by_level()
        return (
            f"AuditLog(total={len(self._events)}, "
            f"info={counts['info']}, "
            f"warning={counts['warning']}, "
            f"error={counts['error']})"
        )

    def log(self, level, event_type, message,
        client_id=None, transaction_id=None, **meta):
        """Записать событие в аудит"""
        
        if level not in self.ALLOWED_LEVELS:
            raise InvalidOperationError(f'Недопустимый уровень: {level}. Разрешены: {", ".join(self.ALLOWED_LEVELS)}'
            )
        
        if not isinstance(event_type, str) or not event_type.strip():
            raise InvalidOperationError('event_type должен быть непустой строкой')

        if not isinstance(message, str) or not message.strip():
            raise InvalidOperationError('message должен быть непустой строкой')
        
        event = {
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "level": level,
            "event_type": event_type.strip(),
            "message": message.strip(),
            "client_id": client_id,
            "transaction_id": transaction_id,
            "meta": meta,
        }

        self._events.append(event)

        if self._file_path:
            self._write_to_file(event)

        return event

    def _write_to_file(self, event):
        """Записать событие в файл"""
        try:
            with open(self._file_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(event, ensure_ascii=False) + "\n")
        except OSError as e:
            print(f"[AuditLog] Ошибка записи в файл: {e}")

    def get_events(self, level=None, client_id=None, event_type=None):
        """Получить события с фильтрацией"""
        result = []
        for event in self._events:
            if level is not None and event["level"] != level:
                continue
            if client_id is not None and event["client_id"] != client_id:
                continue
            if event_type is not None and event["event_type"] != event_type:
                continue
            result.append(event)
        return result

    def get_suspicious(self):
        return [e for e in self._events
                if e["level"] in (self.LEVEL_WARNING, self.LEVEL_ERROR)]

    def get_error_stats(self):
        """Статистика ошибок по типам"""
        stats = {}
        for e in self._events:
            if e["level"] != self.LEVEL_ERROR:
                continue
            key = e["meta"].get("error_type", e["event_type"])
            stats[key] = stats.get(key, 0) + 1
        return stats

    def count_by_event_type(self):
        """Сколько событий каждого типа"""
        counts = {}
        for e in self._events:
            counts[e["event_type"]] = counts.get(e["event_type"], 0) + 1
        return counts

    def count_by_level(self):
        """Вернуть счётчик событий по уровням."""
        counts = {level: 0 for level in self.ALLOWED_LEVELS}
        for event in self._events:
            counts[event["level"]] += 1
        return counts

    def clear(self):
        self._events = []

class RiskAnalyzer:
    
    LARGE_AMOUNT = 100000          
    FREQUENT_COUNT = 3              
    FREQUENT_WINDOW = 60            

    RISK_LOW = "low"
    RISK_MEDIUM = "medium"
    RISK_HIGH = "high"

    def __init__(self, bank, audit_log=None):
        self.bank = bank
        self.audit_log = audit_log
        self._history = {}

    def _check_large_amount(self, transaction):
        """Крупная сумма"""
        if transaction.amount >= self.LARGE_AMOUNT:
            return (2, f'Крупная сумма: {transaction.amount} {transaction.currency}')
        return (0, None)

    def _check_frequent_operations(self, transaction):
        """Частые операции"""
        
        if transaction.sender_id is None:
            return (0, None)

        client_id = self._find_client_by_account(transaction.sender_id)
        if client_id is None:
            return (0, None)

        history = self._history.get(client_id, [])
        now = datetime.now()
        window_start = now - timedelta(minutes=self.FREQUENT_WINDOW)

        recent = [trans for ts, trans in history if ts >= window_start]
        if len(recent) >= self.FREQUENT_COUNT:
            return (2, f'Частые операции: {len(recent)} за {self.FREQUENT_WINDOW} мин')
        return (0, None)

    def _find_client_by_account(self, account_id):
        """Найти client_id по ID счёта."""
        for client in self.bank._clients.values():
            if account_id in client.account_ids:
                return client.client_id
        return None

    def _check_new_receiver(self, transaction):
        """Поиск перевода на новый счет"""
        if transaction.transaction_type != Transaction.TYPE_TRANSFER:
            return (0, None)
        if transaction.receiver_id is None:
            return (0, None)

        client_id = self._find_client_by_account(transaction.sender_id)
        if client_id is None:
            return (0, None)

        history = self._history.get(client_id, [])
        for _, past in history:
            if past.receiver_id == transaction.receiver_id:
                return (0, None) # не новый

        return (1, f'Перевод на новый счёт: {transaction.receiver_id}')

    def _check_night_operation(self, transaction):
        """Операция ночью"""
        created = transaction.created_at.time()
        if time(0, 0) <= created < time(5, 0):
            return (1, f'Ночная операция: {created.strftime("%H:%M")}')
        return (0, None)

    def get_client_profile(self, client_id):
        """Риск-профиль клиента"""
        history = self._history.get(client_id, [])
        if not history:
            return {
                "transaction_count": 0,
                "unique_receivers": 0,
                "total_amount": 0,
                "risk_level": self.RISK_LOW,
            }

        transactions = [t for _, t in history]
        receivers = {t.receiver_id for t in transactions if t.receiver_id}
        total = sum(t.amount for t in transactions)

        if total >= self.LARGE_AMOUNT or len(transactions) >= self.FREQUENT_COUNT * 2:
            level = self.RISK_HIGH
        elif len(transactions) >= self.FREQUENT_COUNT or total >= self.LARGE_AMOUNT // 2:
            level = self.RISK_MEDIUM
        else:
            level = self.RISK_LOW

        return {
            "transaction_count": len(transactions),
            "unique_receivers": len(receivers),
            "total_amount": total,
            "risk_level": level,
        }

    def analyze(self, transaction):
        """Анализ транзакции"""
        if not isinstance(transaction, Transaction):
            raise InvalidOperationError( f'Ожидается Transaction, получено {type(transaction).__name__}')

        checks = [
            self._check_large_amount(transaction),
            self._check_frequent_operations(transaction),
            self._check_new_receiver(transaction),
            self._check_night_operation(transaction),
        ]

        score = sum(weight for weight, _ in checks)
        reasons = [reason for _, reason in checks if reason]

        if score >= 3:
            level = self.RISK_HIGH
        elif score >= 1:
            level = self.RISK_MEDIUM
        else:
            level = self.RISK_LOW

        return {
            "level": level,
            "score": score,
            "reasons": reasons,
        }
    
    def register_transaction(self, client_id, transaction):
        """Запись транзакции в историю клиента"""
        if client_id not in self._history:
            self._history[client_id] = []
        self._history[client_id].append((datetime.now(), transaction))

In [ ]:
import random
from datetime import datetime, timedelta

from day1 import (
    BankAccount,
    InvalidOperationError,
    InsufficientFundsError,
    AccountFrozenError,
    AccountClosedError,
)
from day2 import SavingsAccount, PremiumAccount, InvestmentAccount
from day3 import Bank, Client
from day4 import Transaction, TransactionQueue, TransactionProcessor
from day5 import AuditLog, RiskAnalyzer

random.seed(42)   

def section(title):
    print()
    print("=" * 75)
    print(f"  {title}")
    print("=" * 75)


def subsection(title):
    print()
    print(f" ==={title}=== ")

def balance_in_rub(account, rates):
    """Перевести баланс счёта в RUB по курсам"""
    if account._currency not in rates:
        raise InvalidOperationError(f'Нет курса для валюты {account._currency}')
    return account._balance * rates[account._currency]

def client_balance_in_rub(client, bank, rates):
    """Общий баланс клиента в RUB"""
    total = 0.0
    for acc_id in client.account_ids:
        acc = bank._get_account(acc_id)
        total += balance_in_rub(acc, rates)
    return total

def create_bank():
    """Создание банка с клиентами и счетами."""
    bank = Bank("Демо Банк")

    clients_data = [
        ("Иванов Иван Иванович",     30, "ivanov@mail.ru",   "+79991000001", "pass1"),
        ("Петров Пётр Петрович",      25, "petrov@mail.ru",   "+79991000002", "pass2"),
        ("Сидоров Сидор Сидорович",   35, "sidorov@mail.ru",  "+79991000003", "pass3"),
        ("Смирнова Анна Сергеевна",   28, "smirnova@mail.ru", "+79991000004", "pass4"),
        ("Кузнецов Олег Иванович",    40, "kuznetsov@mail.ru", "+79991000005", "pass5"),
        ("Попова Мария Алексеевна",   22, "popova@mail.ru",   "+79991000006", "pass6"),
    ]

    client_ids = []
    for name, age, email, phone, password in clients_data:
        c = Client(name, age, email, phone, password)
        client_ids.append(bank.add_client(c))

    # Счета
    accounts = {}

    # Иванов: basic + savings
    accounts["ivanov_basic"]   = bank.open_account(client_ids[0], "basic", balance=100000)
    accounts["ivanov_savings"] = bank.open_account(client_ids[0], "savings",
                                                    balance=50000, min_balance=10000)
    # Иванов: USD 
    accounts["ivanov_usd"] = bank.open_account(
        client_ids[0], "basic", balance=1000, currency="USD"
    )   

    # Петров: basic
    accounts["petrov_basic"]   = bank.open_account(client_ids[1], "basic", balance=50000)

    # Сидоров: premium + basic
    accounts["sidorov_premium"] = bank.open_account(client_ids[2], "premium",
                                                     balance=500000,
                                                     overdraft_limit=100000,
                                                     operation_fee=100)
    accounts["sidorov_basic"]   = bank.open_account(client_ids[2], "basic", balance=100000)

    # Смирнова: savings + basic
    accounts["smirnova_savings"] = bank.open_account(client_ids[3], "savings",
                                                      balance=200000, min_balance=50000)
    accounts["smirnova_basic"]   = bank.open_account(client_ids[3], "basic", balance=30000)

    # Кузнецов: premium + investment
    accounts["kuznetsov_premium"] = bank.open_account(client_ids[4], "premium",
                                                       balance=1000000,
                                                       overdraft_limit=200000,
                                                       operation_fee=200)
    accounts["kuznetsov_invest"] = bank.open_account(client_ids[4], "investment",
                                                      balance=200000,
                                                      portfolio={"stocks": 100000,
                                                                 "bonds": 50000,
                                                                 "etf": 50000})

    # Попова: basic
    accounts["popova_basic"] = bank.open_account(client_ids[5], "basic", balance=10000)

    return bank, client_ids, accounts

def generate_transactions(bank, accounts):
    """Сгенерировать 40 транзакций"""
    transactions = []
    acc_list = list(accounts.values())

    for i in range(40):
        kind = random.choices(
            ["normal", "error", "suspicious"],
            weights=[0.6, 0.2, 0.2],
        )[0]

        if kind == "normal":
            t = _make_normal(acc_list)
        elif kind == "error":
            t = _make_error(acc_list)
        else:
            t = _make_suspicious(acc_list)

        transactions.append(t)

    return transactions

def _make_normal(acc_list):
    """Обычный перевод между случайными счетами"""
    sender = random.choice(acc_list)
    receiver = random.choice([a for a in acc_list if a is not sender])

    amount = random.randint(100, 5000)

    return Transaction(
        "transfer", amount, "RUB",
        sender_id=sender.account_id,
        receiver_id=receiver.account_id,
    )

def _make_error(acc_list):
    """Транзакция с ощибкой"""
    sender = random.choice(acc_list)
    receiver = random.choice([a for a in acc_list if a is not sender])

    amount = sender._balance + random.randint(10000, 100000) # делаем транзакцию > баланса

    return Transaction(
        "transfer", amount, "RUB",
        sender_id=sender.account_id,
        receiver_id=receiver.account_id,
    )

def _make_suspicious(acc_list):
    """Подозрительная (крупная сумма)"""
    sender = random.choice(acc_list)
    receiver = random.choice([a for a in acc_list if a is not sender])

    amount = random.randint(150000, 500000)

    return Transaction(
        "transfer", amount, "RUB",
        sender_id=sender.account_id,
        receiver_id=receiver.account_id,
    )

def _find_client_id(bank, transaction):
    """Найти client_id по счёту отправителя/получателя."""
    account_id = transaction.sender_id or transaction.receiver_id
    if account_id is None:
        return None
    for client in bank._clients.values():
        if account_id in client.account_ids:
            return client.client_id
    return None

def process_transactions(bank, transactions, audit, analyzer, processor):
    """Обработка всех транзакции с проверкой рисков"""
    queue = TransactionQueue()

    subsection("Добавление в очередь")
    for t in transactions:
        queue.add(t)
    print(f"В очереди: {len(queue)} транзакций")
    print(f"   {queue}")

    subsection("Проверка риска и выполнение")

    results = {
        "completed": [],
        "failed": [],
        "blocked": [],
    }

    while not queue.is_empty():
        t = queue.get_next()
        if t is None:
            break

        risk = analyzer.analyze(t)

        # Блокировка для high risk
        if risk["level"] == analyzer.RISK_HIGH:
            t.mark_failed(f'Заблокировано: high risk')
            audit.log("error", "transaction_blocked",
                      f'Заблокирована: {t.transaction_id}',
                      transaction_id=t.transaction_id,
                      risk_score=risk["score"],
                      reasons=risk["reasons"])
            results["blocked"].append(t)
            continue

        # Warning для medium
        if risk["level"] == analyzer.RISK_MEDIUM:
            audit.log("warning", "risk_detected",
                      f'Средний риск: {t.transaction_id}',
                      transaction_id=t.transaction_id,
                      risk_score=risk["score"],
                      reasons=risk["reasons"])

        # Выполнение
        processor.execute(t)

        if t.status == "completed":
            audit.log("info", "transaction_completed",
                      f'Выполнена: {t.transaction_id}',
                      transaction_id=t.transaction_id,
                      amount=t.amount)
            cid = _find_client_id(bank, t)
            if cid is not None:
                analyzer.register_transaction(cid, t)
            results["completed"].append(t)
        elif t.status == "failed":
            audit.log("error", "transaction_failed",
                      f'Не успешно: {t.failure_reason}',
                      transaction_id=t.transaction_id,
                      error_type="operation_error",
                      error=t.failure_reason)
            results["failed"].append(t)

    return results

def show_client_scenario(bank, client_id, audit, rates):
    """Показать сценарий для клиента."""
    client = bank._get_client(client_id)

    subsection(f"Сценарий клиента: {client.full_name}")
    print(f"   ID:     {client.client_id}")
    print(f"   Возраст: {client.age}")
    print(f"   Статус: {client.status}")

    
    print(f"\n   Счета ({len(client.account_ids)}):")
    total = 0
    for acc_id in client.account_ids:
        acc = bank._get_account(acc_id)
        bal_rub = balance_in_rub(acc, rates)
        total += bal_rub
        print(f"      {acc.__class__.__name__:18s} "
              f"...{acc_id[-4:]}  "
              f"{acc._balance} {acc._currency} "
              f"({bal_rub} RUB)  "
              f"[{acc._status}]")
    print(f"   Итого: {total} RUB")

    
    events = audit.get_events(client_id=client_id)
    print(f"\n   События в аудите: {len(events)}")
    for ev in events[:5]:  
        print(f"      [{ev['level']}] {ev['event_type']} {ev['message']}")

    # Подозрительные
    suspicious = [ev for ev in events
                  if ev["level"] in ("warning", "error")]
    print(f"\n   Подозрительных: {len(suspicious)}")
    for ev in suspicious[:3]:
        print(f"{ev['message']}")

def show_reports(bank, audit, results, rates):
    """Финальные отчёты"""

    subsection("Топ-3 клиента по балансу в RUB")
    ranking = []
    for client in bank._clients.values():
        total_rub = client_balance_in_rub(client, bank, rates)
        ranking.append((client, total_rub))
    ranking.sort(key=lambda pair: pair[1], reverse=True)

    for i, (client, total) in enumerate(ranking[:3], 1):
        print(f"   {i}. {client.full_name}  {total} RUB")

    subsection("Общий баланс банка в RUB")
    total_rub = sum(
        balance_in_rub(acc, rates)
        for acc in bank._accounts.values()
    )
    print(f"   {total_rub:.2f} RUB")

    
    subsection("Аудит")
    print(f"   Всего событий: {len(audit)}")
    print(f"   По уровням:    {audit.count_by_level()}")
    print(f"   По типам:      {audit.count_by_event_type()}")

    subsection("Статистика ошибок")
    errors = audit.get_error_stats()
    if errors:
        for err_type, count in errors.items():
            print(f"   {err_type}: {count}")
    else:
        print("   Ошибок нет")

def main():
    section("КОМПЛЕКСНАЯ ДЕМОНСТРАЦИЯ БАНКОВСКОЙ СИСТЕМЫ")

    section("1. ИНИЦИАЛИЗАЦИЯ")
    bank, client_ids, accounts = create_bank()
    processor = TransactionProcessor(bank)
    print(f"Банк:     {bank.name}")
    print(f"Клиентов: {len(bank._clients)}")
    print(f"Счетов:   {len(bank._accounts)}")

    subsection("Клиенты и счета ")
    for cid in client_ids:
        c = bank._get_client(cid)
        total = client_balance_in_rub(c, bank, processor.rates)
        print(f"   {c.full_name:32s}  счетов: {len(c.account_ids)}  "
            f"баланс: {total:>12.2f} RUB")


    section("2. ТРАНЗАКЦИИ")
    transactions = generate_transactions(bank, accounts)
    print(f"Всего транзакций: {len(transactions)}")

    section("3. ОБРАБОТКА")
    audit = AuditLog(file_path="audit_day6.log")
    analyzer = RiskAnalyzer(bank, audit)
    
    results = process_transactions(bank, transactions, audit, analyzer, processor)

   
    section("4. СЦЕНАРИЙ КЛИЕНТА")
    show_client_scenario(bank, client_ids[2], audit, processor.rates)
    
    section("5. ОТЧЁТЫ")
    show_reports(bank, audit, results, processor.rates)


if __name__ == "__main__":
    main()        



In [ ]:
import matplotlib
matplotlib.use("Agg")   # ← важно! до импорта pyplot
import matplotlib.pyplot as plt
import os
import csv
import json
from day1 import InvalidOperationError

class ReportBuilder:
    def __init__(self, bank, audit, analyzer, processor, output_dir="reports"):
        self.bank = bank
        self.audit = audit
        self.analyzer = analyzer
        self.processor = processor
        self.output_dir = output_dir

        os.makedirs(output_dir, exist_ok=True)

    def export_to_json(self, data, filename):
        """Сохранить в JSON"""
        path = os.path.join(self.output_dir, filename)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        return path

    def export_to_csv(self, rows, filename, headers=None):
        """Сохранить в CSV"""
        path = os.path.join(self.output_dir, filename)

        if not rows:
            with open(path, "w", encoding="utf-8", newline="") as f:
                if headers:
                    writer = csv.writer(f)
                    writer.writerow(headers)
            return path

        if headers is None:
            headers = list(rows[0].keys())

        with open(path, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=headers)
            writer.writeheader()
            writer.writerows(rows)

        return path

    def export_to_txt(self, text, filename):
        """Сохранить строку в текстовый файл."""
        path = os.path.join(self.output_dir, filename)
        with open(path, "w", encoding="utf-8") as f:
            f.write(text)
        return path

    def _format_bank_text(self, data):
        """Текстовое представление отчёта по банку"""
        lines = [
            "=" * 60,
            f"ОТЧЁТ ПО БАНКУ: {data['bank_name']}",
            "=" * 60,
            f"Клиентов:      {data['clients_count']}",
            f"Счетов:        {data['accounts_count']}",
            f"Общий баланс:  {data['total_balance_rub']} RUB",
            "",
            "ТОП-3 КЛИЕНТА:",
        ]
        for i, c in enumerate(data["top_3"], 1):
            lines.append(f"  {i}. {c['full_name']}  {c['total_rub']} RUB")

        lines.append("\nАУДИТ:")
        audit_sum = data["audit_summary"]
        lines.append(f"  Всего событий: {audit_sum['total_events']}")
        lines.append(f"  По уровням:    {audit_sum['by_level']}")
        lines.append(f"  По типам:      {audit_sum['by_type']}")

        return "\n".join(lines)

    def _format_risk_text(self, data):
        """Текстовое представление отчёта по рискам"""
        lines = [
            "=" * 60,
            "ОТЧЁТ ПО РИСКАМ",
            "=" * 60,
            f"Подозрительных событий: {data['suspicious_count']}",
            "",
            "СТАТИСТИКА ОШИБОК:",
        ]
        if data["error_stats"]:
            for err_type, count in data["error_stats"].items():
                lines.append(f"  {err_type}: {count}")
        else:
            lines.append("  Ошибок нет")

        lines.append("\nРИСК-ПРОФИЛИ КЛИЕНТОВ:")
        for c in data["risky_clients"]:
            lines.append(
                f"  {c['full_name']}  "
                f"риск: {c['risk_level']}  "
                f"транзакций: {c['transaction_count']}  "
                f"сумма: {c['total_amount']}"
            )

        return "\n".join(lines)

    def build_client_report(self, client_id, fmt="txt"):
        """Отчёт по клиенту."""
        client = self.bank._get_client(client_id)

        accounts_info = []
        total_rub = 0.0
        for acc_id in client.account_ids:
            acc = self.bank._get_account(acc_id)
            bal_rub = acc._balance * self.processor.rates[acc._currency]
            total_rub += bal_rub
            accounts_info.append({
                "id": acc_id,
                "type": acc.__class__.__name__,
                "balance": acc._balance,
                "currency": acc._currency,
                "balance_rub": round(bal_rub, 2),
                "status": acc._status,
            })

        events = self.audit.get_events(client_id=client_id)
        profile = self.analyzer.get_client_profile(client_id)

        data = {
            "client": {
                "id": client.client_id,
                "full_name": client.full_name,
                "age": client.age,
                "email": client.email,
                "phone": client.phone,
                "status": client.status,
            },
            "accounts": accounts_info,
            "total_rub": round(total_rub, 2),
            "risk_profile": profile,
            "events_count": len(events),
        }

        if fmt == "json":
            path = self.export_to_json(data, f"client_{client_id}.json")
            return path

        if fmt == "txt":
            text = self._format_client_text(data)
            path = self.export_to_txt(text, f"client_{client_id}.txt")
            return path

        raise InvalidOperationError(f'Неизвестный формат: {fmt}')

    def _format_client_text(self, data):
        """Текстовое представление отчёта по клиенту."""
        c = data["client"]
        lines = [
            "=" * 60,
            f"ОТЧЁТ ПО КЛИЕНТУ: {c['full_name']}",
            "=" * 60,
            f"ID:      {c['id']}",
            f"Возраст: {c['age']}",
            f"Email:   {c['email']}",
            f"Телефон: {c['phone']}",
            f"Статус:  {c['status']}",
            "",
            "СЧЕТА:",
        ]
        for acc in data["accounts"]:
            lines.append(
                f"  {acc['type']} ...{acc['id'][-4:]}  "
                f"{acc['balance']} {acc['currency']} "
                f"(≈ {acc['balance_rub']} RUB)  [{acc['status']}]"
            )
        lines.append(f"\nИТОГО: {data['total_rub']} RUB")

        lines.append("\nРИСК-ПРОФИЛЬ:")
        p = data["risk_profile"]
        lines.append(f"  Транзакций:    {p['transaction_count']}")
        lines.append(f"  Получателей:   {p['unique_receivers']}")
        lines.append(f"  Общая сумма:   {p['total_amount']}")
        lines.append(f"  Уровень риска: {p['risk_level']}")

        lines.append(f"\nСобытий в аудите: {data['events_count']}")
        return "\n".join(lines)

    def build_bank_report(self, fmt="txt"):
        """Общий отчёт по банку."""
        clients_info = []
        for client in self.bank._clients.values():
            total_rub = 0.0
            for acc_id in client.account_ids:
                acc = self.bank._get_account(acc_id)
                total_rub += acc._balance * self.processor.rates[acc._currency]

            clients_info.append({
                "client_id": client.client_id,
                "full_name": client.full_name,
                "age": client.age,
                "status": client.status,
                "accounts_count": len(client.account_ids),
                "total_rub": round(total_rub, 2),
            })

        clients_info.sort(key=lambda c: c["total_rub"], reverse=True)

        total_bank_rub = sum(c["total_rub"] for c in clients_info)

        data = {
            "bank_name": self.bank.name,
            "clients_count": len(self.bank._clients),
            "accounts_count": len(self.bank._accounts),
            "total_balance_rub": round(total_bank_rub, 2),
            "clients": clients_info,
            "top_3": clients_info[:3],
            "audit_summary": {
                "total_events": len(self.audit),
                "by_level": self.audit.count_by_level(),
                "by_type": self.audit.count_by_event_type(),
            },
        }

        if fmt == "json":
            return self.export_to_json(data, "bank_report.json")
        if fmt == "txt":
            return self.export_to_txt(self._format_bank_text(data), "bank_report.txt")
        raise InvalidOperationError(f'Неизвестный формат: {fmt}')

    def build_risk_report(self, fmt="txt"):
        """Отчёт по рискам"""
        suspicious = self.audit.get_suspicious()

        risky_clients = []
        for client in self.bank._clients.values():
            profile = self.analyzer.get_client_profile(client.client_id)
            risky_clients.append({
                "client_id": client.client_id,
                "full_name": client.full_name,
                "risk_level": profile["risk_level"],
                "transaction_count": profile["transaction_count"],
                "total_amount": profile["total_amount"],
            })
        risky_clients.sort(
            key=lambda c: {"high": 3, "medium": 2, "low": 1}[c["risk_level"]],
            reverse=True,
        )

        data = {
            "suspicious_events": suspicious,
            "suspicious_count": len(suspicious),
            "error_stats": self.audit.get_error_stats(),
            "risky_clients": risky_clients,
        }

        if fmt == "json":
            return self.export_to_json(data, "risk_report.json")
        if fmt == "txt":
            return self.export_to_txt(self._format_risk_text(data), "risk_report.txt")
        raise InvalidOperationError(f'Неизвестный формат: {fmt}')

    def chart_balance_pie(self, filename="chart_pie.png"):
        """Круговая"""
        labels = []
        values = []

        for client in self.bank._clients.values():
            total = 0.0
            for acc_id in client.account_ids:
                acc = self.bank._get_account(acc_id)
                total += acc._balance * self.processor.rates[acc._currency]
            if total > 0:
                labels.append(client.full_name)
                values.append(total)

        fig, ax = plt.subplots(figsize=(10, 7))
        ax.pie(values, labels=labels, autopct="%1.1f%%", startangle=90)
        ax.set_title("Распределение баланса по клиентам RUB")

        path = os.path.join(self.output_dir, filename)
        plt.savefig(path, bbox_inches="tight")
        plt.close(fig)
        return path

    def chart_top_clients_bar(self, filename="chart_bar.png", top_n=5):
        """Столбчатая"""
        data = []
        for client in self.bank._clients.values():
            total = 0.0
            for acc_id in client.account_ids:
                acc = self.bank._get_account(acc_id)
                total += acc._balance * self.processor.rates[acc._currency]
            data.append((client.full_name, total))

        data.sort(key=lambda x: x[1], reverse=True)
        data = data[:top_n]

        names = [d[0] for d in data]
        values = [d[1] for d in data]

        fig, ax = plt.subplots(figsize=(10, 6))
        bars = ax.bar(names, values, color="steelblue")
        ax.set_title(f"Топ-{top_n} клиентов по балансу")
        ax.set_ylabel("Баланс, RUB")
        plt.xticks(rotation=30, ha="right")

        
        for bar, val in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{val:,.0f}",
                ha="center", va="bottom", fontsize=9,
            )

        path = os.path.join(self.output_dir, filename)
        plt.savefig(path, bbox_inches="tight")
        plt.close(fig)
        return path

    def chart_balance_history(self, filename="chart_history.png"):
        """Граф движения баланса"""
        events = self.audit.get_events(event_type="transaction_completed")

        if not events:
            print("Нет данных для графика истории")
            return None

        events.sort(key=lambda e: e["timestamp"])

        timestamps = []
        cumulative = []
        total = 0.0
        for e in events:
            amount = e["meta"].get("amount", 0)
            total += amount
            timestamps.append(e["timestamp"])
            cumulative.append(total)

        fig, ax = plt.subplots(figsize=(12, 6))
        ax.plot(range(len(timestamps)), cumulative, marker="o", color="green")
        ax.set_title("Накопленный объём выполненных транзакций RUB")
        ax.set_xlabel("Номер транзакции")
        ax.set_ylabel("Накопленный объём, RUB")
        ax.grid(True, alpha=0.3)

        path = os.path.join(self.output_dir, filename)
        plt.savefig(path, bbox_inches="tight")
        plt.close(fig)
        return path

    def save_charts(self):
        return {
            "pie": self.chart_balance_pie(),
            "bar": self.chart_top_clients_bar(),
            "history": self.chart_balance_history(),
        }

    